In [ ]:
# Aws Lambda Code

In [3]:
import json
import boto3
client=boto3.client('s3')

bucketname="siddhesh-amzn-bucket"

def lambda_handler(event, context):
    response=client.list_objects(Bucket="{}".format(bucketname))
    print(response['Contents'],[0]) 
# Content of buckets code 

In [ ]:
#making seprate files of each department

In [ ]:
import json
import csv
import boto3

client = boto3.client('s3')

bucketName = 'sj-source-11'
filename = 'Dept/employee_data.csv'
local_input = '/tmp/employee_data.csv'

def download_file_from_s3():
    client.download_file(bucketName, filename,local_input)

def read_emp_csv():
    with open(local_input, 'r') as file:
        reader = csv.DictReader(file)
        return list(reader)

def filter_by_dept(file_read):
    dept_data = {}
    for row in file_read:
        dept = row['department']

        if dept not in dept_data:
            dept_data[dept] = []
        dept_data[dept].append(row)
    return dept_data

def write_dept_wise_file(filter_dept):
    created_files = []
    for dept, records in filter_dept.items():
        FileName = '/tmp/{}.csv'.format(dept)

        with open(FileName, 'w', newline = '') as fp:
            writer = csv.DictWriter(fp, fieldnames = records[0].keys())
            writer.writeheader()
            writer.writerows(records)
        created_files.append((dept, FileName))
    return created_files

def upload_files(files):
    for dept,path in files:
        s3_key = 'Dept/{}.csv'.format(dept)
        client.upload_file(path, bucketName, s3_key)

def lambda_handler(event, context):
    download_file_from_s3()
    file_read = read_emp_csv()
    filter_dept = filter_by_dept(file_read)
    files = write_dept_wise_file(filter_dept)
    upload_files(files)
    print("Department files created Successfully")

In [ ]:
#from source bucket to destination bucket

In [ ]:
import json
import boto3
client = boto3.client('s3')

bucketname = 'sj-source-11'
destination = 'sj-destination-11'

def copy_to_destination(filename):
    response = client.copy_object(Bucket='{}'.format(destination),CopySource='/{}/{}'.format(bucketname,filename),Key='{}'.format(filename))

def delete_from_source(filename):
    response = client.delete_object(Bucket='{}'.format(bucketname),Key='{}'.format(filename))



def lambda_handler(event, context):
    response = client.list_objects(Bucket='{}'.format(bucketname))
    for record in response['Contents']:
        filename = record['Key']
        copy_to_destination(filename)
        delete_from_source(filename)
    print("All files are moved to destination bucket")

In [ ]:
#Mysql to S3 Using Lambda

In [ ]:
import json
import pymysql
import boto3
import csv
host = 'database-1.c1406kkmw3qe.ap-south-1.rds.amazonaws.com'
port = 3306
username = 'admin'
password = 'admin#2026'
database = 'emp'
bucketname = 'aws-task-offline11'

client = boto3.client('s3')


def connect_to_db():
    try:

        connection = pymysql.connect(host=host,
                                    user=username,
                                    password=password,
                                    port=port,
                                    db=database,
                                )
        return connection
    except Exception as e:
        print(e)

def data_read_from_db(conn):
    try:
        with conn.cursor() as cur:
            cur.execute("select * from Students1")
            data = cur.fetchall()

        return data
    except Exception as e:
        print(e)

def write_data_to_tmp(data):
    try:
        with open("/tmp/student.csv","w") as fp:
            w =  csv.writer(fp)
            w.writerow(["id","name","age","city"])
            w.writerows(data)
    except Exception as e:
        print(e)


def upload_to_s3():
    client.upload_file("/tmp/student.csv", bucketname, "student.csv")
   

def lambda_handler(event, context):
    conn = connect_to_db()
    data = data_read_from_db(conn)
    write_data_to_tmp(data)
    upload_to_s3()
    print("done")

In [ ]:
#MSSQL to s3 

In [ ]:
import json
import boto3
import pymssql
import csv

server = "database-1.c1406kkmw3qe.ap-south-1.rds.amazonaws.com"
user = "admin"
password = "admin#2026"
port = 1433
database = "collage"
bucketname = "aws-task-offline11"

client = boto3.client("s3")

def conn_to_db():
    try:
        conn = pymssql.connect(
            server=server, user=user, password=password, port=port, database=database
        )
        return conn
    except Exception as e:
        print(e)

def data_read_from_db(conn):
    try:
        with conn.cursor() as cur:
            cur.execute("select * from students")
            data = cur.fetchall()
        return data
    except Exception as e:
        print(e)

def write_data_to_tmp(data):
    try:
        with open("/tmp/students.csv", "w") as fp:
            w = csv.writer(fp)
            w.writerow(["sid", "sname", "smarks"])
            w.writerows(data)
    except Exception as e:
        print(e)

def upload_to_s3():
    client.upload_file("/tmp/students.csv", bucketname, "students.csv")

def lambda_handler(event, context):
    conn = conn_to_db()
    data= data_read_from_db(conn)
    write_data_to_tmp(data)
    upload_to_s3()
    print("done")